## 03_Agent实战项目1

### 导入任何一个你可以使用的模型

In [1]:
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat",
                     api_key = '',
                     base_url = 'https://api.deepseek.com')

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


## 导入数据

In [2]:
import numpy as np
import pandas as pd

telco = pd.read_csv('telco_data.csv')
pd.set_option('max_colwidth',50)

telco.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 导入工具并验证

In [3]:
# LangChain 内置代码解释器工具功能
from langchain_experimental.tools import PythonAstREPLTool

ast = PythonAstREPLTool(locals={"telco": telco}) #把已有的dataframe导入工具上下文
ast_ans = ast.invoke("telco['SeniorCitizen'].mean()") #给定代码，让工具执行

C:\Users\yanchen\AppData\Local\Temp\ipykernel_32060\2562297597.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools import PythonAstREPLTool


In [4]:
# 手工验证是否正确
res = telco['SeniorCitizen'].mean()

print(f'Agent 计算结果：{ast_ans}')
print(f'实际结果：{res}')

Agent 计算结果：0.1621468124378816
实际结果：0.1621468124378816


## 编写对查询生成代码公式的 Agent

In [5]:
# 定义第一个 Agent 的输出格式

from pydantic import BaseModel, Field

class write_formula_format(BaseModel):
    """对用户的数据查询需求编写对应公式"""
    search_formula: str = Field(..., description="对用户的数据查询需求编写对应公式，只允许使用 pandas 和内置库。代码，不返回其他内容")

In [6]:
# 编写对查询生成代码公式的 Agent

from langchain.agents import create_agent
from langchain.tools import tool

@tool
def write_formula(query):
    """
    对用户的需求 query，编写对应查询计算的代码公式
    :param query: str, 必填 - 用户输入的查询需求
    :return:search_formula: str, 对用户的数据查询需求编写对应公式，只允许使用 pandas 和内置库。代码，不返回其他内容
    """
    system_prompt = f"""
    你是一位专业的数据分析师，你会耐心思考然后解答用户提出的问题：
    1. 你可以访问一个名为 `telco` 的 pandas 数据表，你可以使用telco.head().to_markdown() 查看数据集的基本信息
    2. 请根据用户提出的问题，编写 Python 代码来回答。记住只返回代码，不返回其他内容，而且只允许使用 pandas 和内置库。
    """
    
    
    conversation = [
        {"role": "system", "content": '''
        1. 你可以访问一个名为 `telco` 的 pandas 数据表，你可以使用telco.head().to_markdown() 查看数据集的基本信息
        2. 首先，请根据用户提出的问题，编写 Python 代码来回答。记住只返回代码，不返回其他内容，而且只允许使用 pandas 和内置库
        3. 仅返回计算需要的代码公式'''},
        
        {"role": "user", "content": query},
        ]
    
    f_agent = create_agent(model = model,
                           tools = [ast],
                           system_prompt = system_prompt,
                           response_format = write_formula_format
                          )
    
    formula_res = f_agent.invoke({"messages":conversation})
    
    search_formula = formula_res['structured_response'].search_formula
    
    return search_formula

## 编写 Agent 提示词

In [7]:
system_prompt = f"""
你是一位专业的数据分析师，你会耐心思考然后解答用户提出的问题：
1. 你可以访问一个名为 `telco` 的 pandas 数据表，你可以使用telco.head().to_markdown() 查看数据集的基本信息
2. 首先，请根据用户提出的问题，编写 Python 代码来回答。记住只返回代码，不返回其他内容，而且只允许使用 pandas 和内置库。
3. 将生成的代码送入 ast（PythonAstREPLTool）工具进行运行解读，最后返回结果。
"""

## 定义输出格式，创建Agent

In [8]:
class res_format(BaseModel):
    """对用户的查query进行回答，分别为用户问题，计算公式与计算结果"""
    query: str = Field(..., description="对用户的 query 进行精炼总结")
    formula: str = Field(..., description="计算查询的 python 代码公式")
    result: str = Field(..., description="根据formula计算的结果")


agent = create_agent(model = model,
                     tools = [ast, write_formula],
                     system_prompt = system_prompt,
                     response_format = res_format
                    )

## 对 df 表格进行数据分析

In [9]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "我有一张表，名为 telco，请帮我计算 MonthlyCharges 字段的均值。"}]}
)

<class 'pandas.core.frame.DataFrame'>


In [10]:
print(response['structured_response'].query)
print(response['structured_response'].formula)
print(response['structured_response'].result)

计算telco表中MonthlyCharges字段的均值
telco['MonthlyCharges'].mean()
64.76169246059918


In [11]:
for i in response['messages']:
    print(i.__class__)

<class 'langchain_core.messages.human.HumanMessage'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>
<class 'langchain_core.messages.ai.AIMessage'>
<class 'langchain_core.messages.tool.ToolMessage'>


In [12]:
for i in response['messages']:
    print(i, end = '\n\n')

content='我有一张表，名为 telco，请帮我计算 MonthlyCharges 字段的均值。' additional_kwargs={} response_metadata={} id='fd9aea14-8ab3-42fa-9e11-604448603e62'

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 676, 'total_tokens': 741, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 676}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ecb4bb10-a8b8-4b39-b6f9-7bd2b269ea45', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ed503-28d1-71e3-a50b-477740704cf6-0' tool_calls=[{'name': 'python_repl_ast', 'args': {'query': 'import pandas as pd\n# 先查看telco数据表的基本信息\nprint(telco.head().to_markdown())\n'}, 'id': 'call_00_7YIqFnVnLebPrSw5u2W31106', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 676, '

In [13]:
response

{'messages': [HumanMessage(content='我有一张表，名为 telco，请帮我计算 MonthlyCharges 字段的均值。', additional_kwargs={}, response_metadata={}, id='fd9aea14-8ab3-42fa-9e11-604448603e62'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 676, 'total_tokens': 741, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 676}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ecb4bb10-a8b8-4b39-b6f9-7bd2b269ea45', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ed503-28d1-71e3-a50b-477740704cf6-0', tool_calls=[{'name': 'python_repl_ast', 'args': {'query': 'import pandas as pd\n# 先查看telco数据表的基本信息\nprint(telco.head().to_markdown())\n'}, 'id': 'call_00_7YIqFnVnLebPrSw5u2W31106', 'type': 'tool_call'}], invalid_tool